In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["NUMEXPR_NUM_THREADS"] = "4"

import torch
torch.set_num_threads(4)
torch.set_num_interop_threads(1)

In [2]:
import sys

import torch

import pandas as pd

import pm4py

from config.feature_config import FeatureConfig
from config.ga_config import GAConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from ga_search.search import CounterfactualGA

from process.engine import ProcessModelConstraintEngine
from process.experimenter import ExperimentHandler

### --- Load Dataset & Models ---

In [3]:
set_seed(seed=777)

In [4]:
df = pd.read_excel(
    "../../../data/bpic17.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "lifecycle:transition": "string",
        "org:resource": "string",
        "case:LoanGoal": "string",
        "case:ApplicationType": "string",
        "Accepted": "string",
        "Selected": "string",
        "case:RequestedAmount": "float32",
        "FirstWithdrawalAmount": "float32",
        "NumberOfTerms": "float32",
        "MonthlyCost": "float32",
        "CreditScore": "float32",
        "OfferedAmount": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [5]:
df.head(20)

,case:concept:name,time:timestamp,Accepted,CreditScore,FirstWithdrawalAmount,MonthlyCost,NumberOfTerms,OfferedAmount,Selected,case:ApplicationType,case:LoanGoal,case:RequestedAmount,concept:name,lifecycle:transition,org:resource,time_delta
0,Application_1000086665,2016-08-03 15:57:21.673,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,A_Create Application,complete,User_1,0.000000e+00
1,Application_1000086665,2016-08-03 15:57:21.734,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,A_Submitted,complete,User_1,6.100000e-02
2,Application_1000086665,2016-08-03 15:58:28.299,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,A_Concept,complete,User_1,6.656500e+01
3,Application_1000086665,2016-08-05 13:57:07.419,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,A_Accepted,complete,User_5,1.655191e+05
4,Application_1000086665,2016-08-05 13:59:57.320,True,0.0,5000.0,241.279999,22.0,5000.0,False,New credit,"Other, see explanation",5000.0,O_Create Offer,complete,User_5,1.699010e+02
5,Application_1000086665,2016-08-05 13:59:58.162,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,O_Created,complete,User_5,8.420000e-01
6,Application_1000086665,2016-08-05 14:01:23.264,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,O_Sent (mail and online),complete,User_5,8.510200e+01
7,Application_1000086665,2016-08-05 14:01:23.288,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,A_Complete,complete,User_5,2.400000e-02
8,Application_1000086665,2016-09-05 06:00:36.710,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,A_Cancelled,complete,User_1,2.649554e+06
9,Application_1000086665,2016-09-05 06:00:36.829,NA,0.0,0.0,0.000000,0.0,0.0,NA,New credit,"Other, see explanation",5000.0,O_Cancelled,complete,User_1,1.190000e-01


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [7]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)

In [8]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['Accepted', 'CreditScore', 'FirstWithdrawalAmount', 'MonthlyCost', 'NumberOfTerms', 'OfferedAmount', 'Selected', 'case:ApplicationType', 'case:LoanGoal', 'case:RequestedAmount', 'concept:name', 'lifecycle:transition', 'org:resource', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
case:LoanGoal                  categorical    case     yes    ['Boat', 'Business goal', 'Car', ...]    N/A        data_derived        
case:ApplicationType           categorical    case     yes    ['Limit raise', 'New credit']            N/A        data_derived        
Accepted         

In [9]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

In [10]:
model = ProcessLSTM.load(
    path = "../pretrained_models/"
)

In [11]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

### --- Process Constraints ---

In [12]:
engine = ProcessModelConstraintEngine.load(
    path = "../pretrained_models/"
)

In [13]:
engine.parallel_sets

[{'A_Incomplete', 'A_Validating', 'O_Returned'}]

In [14]:
engine.branching_sets

[{'A_Complete',
  'A_Incomplete',
  'A_Validating',
  'O_Create Offer',
  'O_Created',
  'O_Returned',
  'O_Sent (mail and online)'},
 {'O_Create Offer', 'O_Created', 'O_Sent (mail and online)'},
 {'A_Denied', 'O_Refused'}]

### --- Experiments Generation ---

In [15]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic17-cf_seed777_experiments_ga_ablated_output.txt", console=False)

In [16]:
generator = ExperimentHandler(
    constraint_engine=engine,
)

In [17]:
exp_df_sin, metadata_sin = ExperimentHandler.load("../experiments/cf_generated_experiments_single_desired")
print("Mined using parameters:", metadata_sin["parameters"])

### --- Counterfactuals ---

In [18]:
ga_config = GAConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=0.0,
)
ga_config.validate()

cf_GA = CounterfactualGA(
    ga_config=ga_config,
    feature_config=feature_config,
    model_wrapper=model_wrapper
)

In [19]:
results_sin = generator.run_experiment_df(
    cf_method=cf_GA,
    technique="GA_Ablated_single_desired_seed777",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/500 [00:00<?, ?case/s]

In [20]:
results_sin

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,Application_509718181,7,1,0,0.320143,0.222638,0.417647,0.425000,0.764706,...,0.393941,0.764706,0.164774,0.235294,0.094255,0.229167,0.0,0.970932,0.0,1.0
1,0,Application_1424170205,9,1,0,0.334409,0.218818,0.450000,0.408333,0.730952,...,0.300966,0.285714,0.113466,0.117647,0.109284,0.187500,0.0,0.000000,0.0,0.0
2,0,Application_542586744,10,1,3,0.353063,0.226714,0.479412,0.446875,0.826087,...,0.293388,0.826087,0.147555,0.235294,0.059816,0.145833,0.0,0.972817,0.0,1.0
3,0,Application_1690291723,11,1,2,0.349071,0.212847,0.485294,0.452083,0.840000,...,0.389103,0.840000,0.180770,0.176471,0.185070,0.208333,0.0,0.925031,0.0,1.0
4,0,Application_135180486,12,1,3,0.375497,0.265700,0.485294,0.490625,0.831481,...,0.587493,0.444444,0.254159,0.352941,0.155378,0.333333,0.0,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
374,48,Application_317628520,22,1,8,0.238861,0.166293,0.311429,0.320588,0.800000,...,0.170196,0.617021,0.072156,0.057143,0.087170,0.098039,0.0,0.926271,0.0,1.0
375,48,Application_2144131288,23,1,5,0.257641,0.142424,0.372857,0.334314,0.918367,...,0.187762,0.918367,0.089723,0.114286,0.065160,0.098039,0.0,0.821748,0.0,1.0
376,48,Application_2057503198,24,1,6,0.258997,0.163709,0.354286,0.348039,0.921569,...,0.270106,0.921569,0.132851,0.200000,0.065702,0.137255,0.0,0.888851,0.0,1.0
377,48,Application_727907012,25,1,3,0.232059,0.116975,0.347143,0.316176,0.817925,...,0.064825,0.490566,0.035413,0.057143,0.013684,0.029412,0.0,0.000000,0.0,0.0


### --- Cleanup ---

In [21]:
sys.stdout = original_stdout
log_file.close()